<a href="https://colab.research.google.com/github/Sarang68/llm-tuning/blob/main/Llama4_FineTuning_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Llama 4 Fine-Tuning with QLoRA on Google Colab

**Stack:** Unsloth + PEFT + TRL + BitsAndBytes + HuggingFace

**Model:** `meta-llama/Llama-4-Scout-17B-16E-Instruct` (17B active params, MoE architecture)

**Technique:** QLoRA — 4-bit quantization + LoRA adapters

**Goal:** Fine-tune Llama 4 Scout on a custom instruction dataset without needing an A100.

---

### ⚙️ Recommended Colab Runtime
- **Runtime → Change runtime type → T4 GPU** (free tier, 15GB)
- For larger batch sizes or longer training: upgrade to **A100 (Colab Pro+)**

### 📋 Prerequisites
1. A [HuggingFace account](https://huggingface.co) with access to Llama 4 (request via Meta's form)
2. HuggingFace token stored in Colab Secrets as `HF_TOKEN`
3. (Optional) Weights & Biases account for experiment tracking

---

In [4]:
%%capture
# Install from Unsloth GitHub — bypasses PyPI version issues
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade transformers>=4.51.0 trl peft bitsandbytes accelerate datasets sentencepiece

## 📦 Step 1: Install Dependencies

In [ ]:
# Install Unsloth (optimized for Colab, handles Llama 4 natively)
# Unsloth reduces VRAM usage by ~60% vs vanilla HuggingFace
!pip install unsloth --quiet

# Core fine-tuning libraries
!pip install -q \
    transformers>=4.51.0 \
    peft>=0.10.0 \
    trl>=0.8.6 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.28.0 \
    datasets \
    huggingface_hub \
    sentencepiece \
    wandb

print("✅ All dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
# Cell 1 — Install + Auto Restart
import subprocess, sys, os

# Install Unsloth
subprocess.run([sys.executable, "-m", "pip", "install",
                "unsloth", "--quiet"], check=True)

# Install remaining libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.51.0",
    "peft>=0.10.0",
    "trl>=0.8.6",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.28.0",
    "datasets",
    "huggingface_hub",
    "sentencepiece",
    "wandb"], check=True)

# Verify installation
import importlib.util
if importlib.util.find_spec("unsloth") is not None:
    print("✅ Unsloth installed successfully")
else:
    print("❌ Unsloth install failed — check output above for errors")

# Auto-restart runtime to pick up new packages
print("\n🔄 Restarting runtime to activate packages...")
os.kill(os.getpid(), 9)

## 🔐 Step 2: Authenticate HuggingFace

> **Colab Secrets Setup:** Go to 🔑 (key icon in left sidebar) → Add secret `HF_TOKEN` → paste your HuggingFace token

In [2]:
from google.colab import userdata
from huggingface_hub import login
import os

# Load token from Colab Secrets (never hardcode tokens!)
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

print("✅ HuggingFace authentication successful")

✅ HuggingFace authentication successful


## 🖥️ Step 3: Check GPU Memory

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 Total VRAM: {total_mem:.1f} GB")
    print(f"📊 Available VRAM: {(total_mem - torch.cuda.memory_allocated(0)/1e9):.1f} GB")
else:
    print("❌ No GPU detected — switch runtime to T4 GPU")

🎮 GPU: Tesla T4
💾 Total VRAM: 15.6 GB
📊 Available VRAM: 15.6 GB


## 🤖 Step 4: Load Llama 4 with QLoRA (4-bit Quantization)

**What happens here:**
- Base model weights are loaded in **4-bit precision** (BitsAndBytes NF4)
- This compresses the model from ~70GB to ~10GB VRAM
- LoRA adapter layers are kept in **16-bit (bfloat16)** for training quality

In [2]:
# Run this cell first to verify everything is in order
from huggingface_hub import whoami, model_info
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# 1. Confirm your identity
me = whoami(token=HF_TOKEN)
print(f"✅ Logged in as: {me['name']}")

# 2. Confirm model is accessible
info = model_info("unsloth/Llama-4-Scout-17B-16E-Instruct", token=HF_TOKEN)
print(f"✅ Model accessible: {info.modelId}")
print(f"✅ Model size: {info.safetensors}")

✅ Logged in as: Sarang68
✅ Model accessible: unsloth/Llama-4-Scout-17B-16E-Instruct
✅ Model size: SafeTensorsInfo(parameters={'BF16': 108641793536}, total=108641793536)


In [4]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME  = "unsloth/Llama-4-Scout-17B-16E-Instruct"
MAX_SEQ_LEN = 1024

# 4-bit quantization config (same as QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,   # Saves extra ~0.4 bits per param
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN
)
tokenizer.pad_token = tokenizer.eos_token

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    token               = HF_TOKEN,
)

# Prepare for LoRA training
model = prepare_model_for_kbit_training(model)
print("✅ Model loaded via HuggingFace (no Unsloth)")
print(f"💾 VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

`rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0
`rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

## 🧩 Step 5: Apply PEFT — Attach LoRA Adapters

**What happens here:**
- The base model weights are **frozen** (no gradients computed)
- Small trainable LoRA matrices are injected into attention + MLP layers
- Only ~1-3% of parameters are actually trained

**Key hyperparameters:**
- `r` = rank of adapter matrices (higher = more capacity, more VRAM)
- `lora_alpha` = scaling factor (typically 2x the rank)
- `target_modules` = which layers to inject adapters into

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r               = 16,           # LoRA rank — try 8 on T4 to save VRAM
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj"       # MLP / FFN layers
    ],
    lora_alpha      = 32,           # Scaling: typically 2x rank
    lora_dropout    = 0.05,         # Light dropout for regularization
    bias            = "none",       # No bias tuning (saves memory)
    use_gradient_checkpointing = "unsloth",  # Reduces VRAM by ~30%
    random_state    = 42,
    use_rslora      = False,        # Set True for Rank-Stabilized LoRA
    loftq_config    = None,
)

# Show trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🎯 Trainable parameters : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"📦 Total parameters     : {total:,}")
print(f"🧊 Frozen parameters    : {total-trainable:,}")

## 📚 Step 6: Prepare Dataset

Using **Alpaca** format — a clean instruction-following dataset.
Replace this with your own domain dataset for production use cases.

**Dataset format expected:**
```json
{"instruction": "...", "input": "...", "output": "..."}
```

In [ ]:
from datasets import load_dataset

# ── Option A: Use public Alpaca dataset (default for this lab) ──
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# ── Option B: Load your own JSONL file ──────────────────────────
# from datasets import load_dataset
# dataset = load_dataset("json", data_files="/content/my_dataset.jsonl", split="train")

# ── Option C: Load from HuggingFace Hub ─────────────────────────
# dataset = load_dataset("your-org/your-dataset", split="train")

print(f"✅ Dataset loaded: {len(dataset)} examples")
print(f"📋 Columns: {dataset.column_names}")
print("\n🔍 Sample entry:")
print(dataset[0])

In [ ]:
# Format dataset using Llama 4 chat template
# Llama 4 uses <|begin_of_text|> ... <|eot_id|> format

LLAMA4_PROMPT_TEMPLATE = """<|begin_of_text|><|header_start|>system<|header_end|>
You are a helpful assistant. Answer the instruction accurately and concisely.<|eot_id|>
<|header_start|>user<|header_end|>
{instruction}
{input}<|eot_id|>
<|header_start|>assistant<|header_end|>
{output}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instr, inp, out in zip(instructions, inputs, outputs):
        input_text = f"\nContext: {inp}" if inp.strip() else ""
        text = LLAMA4_PROMPT_TEMPLATE.format(
            instruction=instr,
            input=input_text,
            output=out
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(format_prompts, batched=True)

# Use a subset for quick experimentation (remove limit for full training)
dataset = dataset.select(range(500))  # Remove this line for full dataset

print(f"✅ Dataset formatted: {len(dataset)} examples")
print("\n🔍 Formatted sample (first 500 chars):")
print(dataset[0]['text'][:500])

## 🏋️ Step 7: Configure and Launch Training

**Key training arguments:**
- `per_device_train_batch_size` → lower if OOM (Out of Memory) errors
- `gradient_accumulation_steps` → compensates for small batch size
- `max_steps` → use instead of epochs for quick experiments
- `learning_rate` → 2e-4 is standard for LoRA fine-tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model            = model,
    tokenizer        = tokenizer,
    train_dataset    = dataset,
    dataset_text_field = "text",
    max_seq_length   = MAX_SEQ_LEN,
    dataset_num_proc = 2,
    packing          = False,      # Set True for short sequences (faster)

    args = TrainingArguments(
        # ── Batch & Memory ──────────────────────────────────────
        per_device_train_batch_size  = 2,   # Lower to 1 if T4 OOM
        gradient_accumulation_steps  = 4,   # Effective batch = 2x4 = 8
        gradient_checkpointing       = True,

        # ── Training Duration ────────────────────────────────────
        max_steps    = 60,          # ~1 min on T4. Use num_train_epochs=3 for full run
        warmup_steps = 5,

        # ── Optimizer ────────────────────────────────────────────
        learning_rate = 2e-4,
        optim         = "adamw_8bit",   # 8-bit optimizer saves ~2GB VRAM
        weight_decay  = 0.01,
        lr_scheduler_type = "linear",

        # ── Precision ─────────────────────────────────────────────
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        # ── Logging & Saving ──────────────────────────────────────
        logging_steps   = 10,
        output_dir      = "/content/llama4-finetuned",
        save_strategy   = "steps",
        save_steps      = 50,
        seed            = 42,

        # ── Optional: WandB tracking ──────────────────────────────
        # report_to = "wandb",
        # run_name  = "llama4-qlora-alpaca",
    ),
)

print("✅ Trainer configured")

In [ ]:
# Show VRAM usage before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"🎮 GPU: {gpu_stats.name}")
print(f"💾 VRAM reserved before training : {start_gpu_memory} GB")
print(f"💾 Total VRAM available          : {max_memory} GB")

# 🚀 Start training
print("\n🚀 Starting training...")
trainer_stats = trainer.train()

# Show VRAM usage after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n📊 Training complete!")
print(f"⏱️  Runtime           : {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"💾 Peak VRAM used     : {used_memory} GB")
print(f"🧩 VRAM for LoRA only : {used_memory_for_lora} GB")

## 🧪 Step 8: Run Inference — Test Your Fine-Tuned Model

In [ ]:
from unsloth.chat_templates import get_chat_template

# Switch model to inference mode (disables dropout, etc.)
FastLanguageModel.for_inference(model)

# ── Your test prompt ─────────────────────────────────────────────
test_prompt = "Explain the difference between LoRA and QLoRA in simple terms."
# ─────────────────────────────────────────────────────────────────

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user",   "content": test_prompt}
]

# Tokenize using the chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids        = inputs,
    max_new_tokens   = 512,
    temperature      = 0.7,
    top_p            = 0.9,
    do_sample        = True,
    use_cache        = True,
)

# Decode and print
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"🙋 Prompt : {test_prompt}")
print(f"\n🤖 Response:\n{response}")

## 💾 Step 9: Save LoRA Adapters (Lightweight — Only ~100MB)

In [ ]:
# ── Option A: Save LoRA adapters only (recommended — small file) ──
ADAPTER_SAVE_PATH = "/content/llama4-lora-adapter"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ LoRA adapters saved to: {ADAPTER_SAVE_PATH}")

# Show saved files
import os
for f in os.listdir(ADAPTER_SAVE_PATH):
    size = os.path.getsize(f"{ADAPTER_SAVE_PATH}/{f}") / 1e6
    print(f"   📄 {f} — {size:.1f} MB")

In [ ]:
# ── Option B: Push LoRA adapters to HuggingFace Hub ───────────────
# Replace with your HuggingFace username

HF_USERNAME   = "your-hf-username"          # ← Change this
REPO_NAME     = "llama4-scout-lora-alpaca"   # ← Change this

model.push_to_hub(
    f"{HF_USERNAME}/{REPO_NAME}",
    token=HF_TOKEN
)
tokenizer.push_to_hub(
    f"{HF_USERNAME}/{REPO_NAME}",
    token=HF_TOKEN
)
print(f"✅ Adapters pushed to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

In [ ]:
# ── Option C: Merge adapters into full model + save as GGUF ───────
# GGUF format = run locally with llama.cpp / Ollama
# Note: Requires more disk space (~10GB)

# model.save_pretrained_gguf(
#     "/content/llama4-gguf",
#     tokenizer,
#     quantization_method="q4_k_m"  # 4-bit quantized GGUF
# )
# print("✅ GGUF model saved")

## 🔁 Step 10: Load Your Fine-Tuned Model Later

This is how you reload and use your LoRA adapters in a new session.

In [ ]:
# Reload base model + attach saved LoRA adapters
from unsloth import FastLanguageModel

model_reload, tokenizer_reload = FastLanguageModel.from_pretrained(
    model_name   = "/content/llama4-lora-adapter",  # Or HuggingFace repo path
    max_seq_length = MAX_SEQ_LEN,
    dtype        = None,
    load_in_4bit = True,
    token        = HF_TOKEN,
)

FastLanguageModel.for_inference(model_reload)
print("✅ Fine-tuned model reloaded successfully")

---

## 📊 Summary: What You Just Did

| Step | Technique | Why It Matters |
|------|-----------|----------------|
| Load model | **QLoRA (4-bit)** | Fits 17B model in ~8GB VRAM instead of ~70GB |
| Attach adapters | **PEFT / LoRA** | Only 1-3% of parameters trained |
| Train | **SFTTrainer + 8-bit Adam** | Supervised fine-tuning with memory-efficient optimizer |
| Save | **Adapter-only checkpoint** | ~100MB vs ~35GB for full model |

---

## 🔧 Troubleshooting Common Issues

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Reduce `per_device_train_batch_size` to 1, `max_seq_length` to 1024 |
| `Model access denied` | Request Llama 4 access at meta.ai, wait for HF approval |
| `Token not found` | Add `HF_TOKEN` to Colab Secrets (key icon in left sidebar) |
| Slow training on T4 | Use `packing=True`, reduce `max_steps` for experimentation |
| `bitsandbytes` error | Ensure CUDA is available — switch to GPU runtime |

---

## 🚀 Next Steps for Production

1. **Scale up**: Move training to GCP Vertex AI with A100 for full dataset runs
2. **Evaluate**: Use `lm-evaluation-harness` or your own benchmark suite
3. **Serve**: Deploy merged model via vLLM or TGI on GKE
4. **Monitor**: Add Langfuse or LangSmith for production observability
5. **Iterate**: Experiment with `r=64`, `rslora=True`, or DPO for alignment

---
*Built for hands-on learning — swap the dataset with your domain data to make it yours.*